### This notebook builds a product recommendation system that returns the top N products based on a user's search.  

Instead of only matching keywords, the system understands what the user really means by using semantic search. This helps provide more relevant and useful recommendations.  

This project shows how we can use modern AI to improve how people search and discover products — a great example of the age of AI 🤖

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
import re
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import os

In [3]:
os.getcwd()

'/content'

In [4]:
working_drirectory = "/content/drive/MyDrive/transformers/Ecommerce/ProductRecommendations"

In [5]:
os.chdir(working_drirectory)

In [6]:
!ls

embeddings.npy			   product_recommendation.ipynb
flipkart_com-ecommerce_sample.csv  semantic_product_recommendation.ipynb
FlipKart_ecommerce.zip


In [7]:
embeddings = np.load("./embeddings.npy")

In [8]:
embeddings.shape

(20002, 768)

In [9]:
df = pd.read_csv("./flipkart_com-ecommerce_sample.csv")

In [10]:
df.head()

,uniq_id,crawl_timestamp,product_url,product_name,product_category_tree,pid,retail_price,discounted_price,image,is_FK_Advantage_product,description,product_rating,overall_rating,brand,product_specifications
0,c2d766ca982eca8304150849735ffef9,2016-03-25 22:59:23 +0000,http://www.flipkart.com/alisha-solid-women-s-c...,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",SRTEH2FF9KEDEFGF,999.0,379.0,"[""http://img5a.flixcart.com/image/short/u/4/a/...",False,Key Features of Alisha Solid Women's Cycling S...,No rating available,No rating available,Alisha,"{""product_specification""=>[{""key""=>""Number of ..."
1,7f7036a6d550aaa89d34c77bd39a5e48,2016-03-25 22:59:23 +0000,http://www.flipkart.com/fabhomedecor-fabric-do...,FabHomeDecor Fabric Double Sofa Bed,"[""Furniture >> Living Room Furniture >> Sofa B...",SBEEH3QGU7MFYJFY,32157.0,22646.0,"[""http://img6a.flixcart.com/image/sofa-bed/j/f...",False,FabHomeDecor Fabric Double Sofa Bed (Finish Co...,No rating available,No rating available,FabHomeDecor,"{""product_specification""=>[{""key""=>""Installati..."
2,f449ec65dcbc041b6ae5e6a32717d01b,2016-03-25 22:59:23 +0000,http://www.flipkart.com/aw-bellies/p/itmeh4grg...,AW Bellies,"[""Footwear >> Women's Footwear >> Ballerinas >...",SHOEH4GRSUBJGZXE,999.0,499.0,"[""http://img5a.flixcart.com/image/shoe/7/z/z/r...",False,Key Features of AW Bellies Sandals Wedges Heel...,No rating available,No rating available,AW,"{""product_specification""=>[{""key""=>""Ideal For""..."
3,0973b37acd0c664e3de26e97e5571454,2016-03-25 22:59:23 +0000,http://www.flipkart.com/alisha-solid-women-s-c...,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",SRTEH2F6HUZMQ6SJ,699.0,267.0,"[""http://img5a.flixcart.com/image/short/6/2/h/...",False,Key Features of Alisha Solid Women's Cycling S...,No rating available,No rating available,Alisha,"{""product_specification""=>[{""key""=>""Number of ..."
4,bc940ea42ee6bef5ac7cea3fb5cfbee7,2016-03-25 22:59:23 +0000,http://www.flipkart.com/sicons-all-purpose-arn...,Sicons All Purpose Arnica Dog Shampoo,"[""Pet Supplies >> Grooming >> Skin & Coat Care...",PSOEH3ZYDMSYARJ5,220.0,210.0,"[""http://img5a.flixcart.com/image/pet-shampoo/...",False,Specifications of Sicons All Purpose Arnica Do...,No rating available,No rating available,Sicons,"{""product_specification""=>[{""key""=>""Pet Type"",..."


In [ ]:
model = SentenceTransformer('all-mpnet-base-v2')

In [33]:
def  get_products(user_input, products_dataframe ,embeddings, embedding_model = model, recommendation_num = 10 ):
  user_input = re.sub(r"[^a-zA-z0-9]", " ", user_input)
  user_embedding_vector = embedding_model.encode([user_input])
  similarities = cosine_similarity(user_embedding_vector, embeddings) # 2D array of similarity scores
  similarities = similarities[0]
  sorted_indices_desc = similarities.argsort()[::-1]
  recommended_products_indexes = sorted_indices_desc[0:recommendation_num + 1]
  recommended_products = products_dataframe.iloc[recommended_products_indexes, [3, 5, 10]]
  return recommended_products


# Testing the Recommendation System

In [35]:
user_input="bluetooth wireless headphones"
print("Recommendations for \"",user_input,"\" are: ")
recommended_products = get_products(user_input, df, embeddings)
recommended_products

Recommendations for " bluetooth wireless headphones " are: 


,product_name,pid,description
5507,Tech Yug BH-503 bluethooth headset Wireless Bl...,ACCEHYWDPRTRMCMS,Key Features of Tech Yug BH-503 bluethooth hea...
8017,LIFE LIKE HBS-730 WITH MIC Wireless Bluetooth ...,ACCEHY86UG6B63AZ,Key Features of LIFE LIKE HBS-730 WITH MIC Wir...
19081,Head Kik Premium Quality Solo2 S460 Wireless B...,ACCEGJPRSJTAGYKV,Key Features of Head Kik Premium Quality Solo2...
9163,Zidane 503TF_BLK Wireless Bluetooth Headset,ACCEGGDGP2KETAD9,Key Features of Zidane 503TF_BLK Wireless Blue...
18936,boom premium quality s-450-p Dynamic Wired Hea...,ACCEJS3ZY9X2GGYW,Key Features of boom premium quality s-450-p D...
18659,GND Wired Earphones Dynamic Handsfree Wired He...,ACCEH8TYZ3MZFV8H,Key Features of GND Wired Earphones Dynamic Ha...
7930,LIFE LIKE S450 3.0 WITH MIC GOOD SOUND QUALITY...,ACCEGZ8GRVUADQ4Y,Key Features of LIFE LIKE S450 3.0 WITH MIC GO...
8007,LIFE LIKE OTP-200 WITH MIC Wireless Bluetooth ...,ACCEHY8AEDYU3W3T,Key Features of LIFE LIKE OTP-200 WITH MIC Wir...
9185,CHKOKKO Earbud Skin Wireless Bluetooth Headset,ACCEGGHE7H4QBGSC,Key Features of CHKOKKO Earbud Skin Wireless B...
18639,THERISE MD0005 Wired Headset,ACCEJ5WM49X2XKDU,Key Features of THERISE MD0005 Wired Headset H...


In [36]:
user_input="Looking for a budget smartphone under 15000 with good camera"
print("Recommendations for \"",user_input,"\" are: ")
recommended_products = get_products(user_input, df, embeddings)
recommended_products

Recommendations for " Looking for a budget smartphone under 15000 with good camera " are: 


,product_name,pid,description
327,Cellbazaar Blackberry 8520 WHITE LCD LCD,MDYEHDH9HMZPANGJ,Key Features of Cellbazaar Blackberry 8520 WHI...
5622,Intel 1.8 GHz LGA 1155 Celeron G460 Processor,PSRD7NKJGFMGAZQG,Buy Intel 1.8 GHz LGA 1155 Celeron G460 Proces...
7003,Huawei HG532D,RTRE93J4UYHVQAKG,Buy Huawei HG532D only for Rs. 1799 from Flipk...
6629,Huawei HG532D: ADSL2+ 300 Mbps Modem With Router,RTRE978WCZ6B8GPW,Buy Huawei HG532D: ADSL2+ 300 Mbps Modem With ...
9989,HP 15-ac121tu (Notebook) (Core i3 (5th Gen)/ 4...,COMEAZ94HWYQTZHZ,Buy HP 15-ac121tu (Notebook) (Core i3 (5th Gen...
6952,ASUS DSL-N10S_B Wireless-N150 ADSL Modem,RTRE4EY9HHDVPVG6,Buy ASUS DSL-N10S_B Wireless-N150 ADSL Modem o...
6255,Huawei WS319 300 Mbps Wireless N Router,RTRE96W36WAJHRCF,Buy Huawei WS319 300 Mbps Wireless N Router on...
1939,EDGE PLUS BODY PANEL FOR SAMSUNG GALAXY S4 950...,MBPEFHRYGGHJMRYW,EDGE PLUS BODY PANEL FOR SAMSUNG GALAXY S4 950...
9988,HP 15-ac116TX (Notebook) (Core i3 (5th Gen)/ 4...,COMEAZ945RHFFGUS,Buy HP 15-ac116TX (Notebook) (Core i3 (5th Gen...
5626,Intel 3.1 GHz LGA 1150 E3-1220 v3 Processor,PSRE2K36QU4U5HBC,Buy Intel 3.1 GHz LGA 1150 E3-1220 v3 Processo...


In [37]:
user_input="Comfortable cotton t-shirts for men"
print("Recommendations for \"",user_input,"\" are: ")
recommended_products = get_products(user_input, df, embeddings)
recommended_products

Recommendations for " Comfortable cotton t-shirts for men " are: 


,product_name,pid,description
5523,Cotton World Men's Solid Casual Linen White Shirt,SHTEJGESKVEMGRYR,Key Features of Cotton World Men's Solid Casua...
10172,Goodkarma Men's Printed Casual Shirt,SHTEGZZCHCEYJJ2J,Key Features of Goodkarma Men's Printed Casual...
10142,Goodkarma Men's Self Design Casual Shirt,SHTEGZZCFCNRPUR7,Key Features of Goodkarma Men's Self Design Ca...
7485,"ARISE Self Design, Printed Men's Henley Green ...",TSHEJQ2FHCZZMUZB,"Key Features of ARISE Self Design, Printed Men..."
10178,Goodkarma Men's Self Design Casual Shirt,SHTEGZZCH49Y3YHY,Key Features of Goodkarma Men's Self Design Ca...
16496,T-shirt Company Full Sleeve Solid Men's Sweats...,SWSEYRJPSRHA3PTV,T-shirt Company Full Sleeve Solid Men's Sweats...
13971,Nod'R Solid Men's Round Neck T-Shirt,TSHEDKKXGFNMZGXR,Nod'R Solid Men's Round Neck T-Shirt (Pack of ...
13849,Nod'R Solid Men's Round Neck T-Shirt,TSHEDKKXFG4R57PZ,Nod'R Solid Men's Round Neck T-Shirt (Pack of ...
9319,The Cotton Company Solid Women's Polo Neck Pin...,TSHEGFWN7RHQBRMU,Key Features of The Cotton Company Solid Women...
10150,Marc N' Park Men's Solid Casual Shirt,SHTEGQCHRXYSRJRE,Key Features of Marc N' Park Men's Solid Casua...


In [38]:
user_input="Non-stick frying pan durable and easy to clean"
print("Recommendations for \"",user_input,"\" are: ")
recommended_products = get_products(user_input, df, embeddings)
recommended_products

Recommendations for " Non-stick frying pan durable and easy to clean " are: 


,product_name,pid,description
1450,NIRLON Classic Pan 25 cm diameter,PTPEJNZ3HJZ3BWHN,Key Features of NIRLON Classic Pan 25 cm diame...
5243,Trinity CELESTA Kadai 250mm Kadhai 1 L,PTPEDDXTMZKMNXHX,Trinity CELESTA Kadai 250mm Kadhai 1 L (Alumin...
1454,NIRLON Classic Kadhai 2.5 L,PTPEJNZ3JY75FCYC,Key Features of NIRLON Classic Kadhai 2.5 L Ni...
4757,Hawkins Futura Hard Anodized Deep-Fry Pan 7.5 ...,PTPDVPEXFUZNDZZH,Buy Hawkins Futura Hard Anodized Deep-Fry Pan ...
4987,Sumeet Quad Kadhai 2 L,PTPE9DXYPQUKBEKF,"Sumeet Quad Kadhai 2 L (Aluminium, Non-stick)\..."
18563,overstockedkitchen Non-Stick Spatula,SPAEJ5U8ZZDBEHNS,overstockedkitchen Non-Stick Spatula (Pack of ...
1458,NIRLON Induction Kadhai 1.5 L,PTPEJPZSEFCE3CZW,Key Features of NIRLON Induction Kadhai 1.5 L ...
4733,Bright Home Appliances Kadhai 2.5 L,PTPE7R2F5GVPTERA,Bright Home Appliances Kadhai 2.5 L (Aluminium...
19322,Whirlpool Affresh Cooktop Cleaner Kitchen Cleaner,KSCEG7CHTWWW4SPG,Key Features of Whirlpool Affresh Cooktop Clea...
1447,oxford Tawa 15 cm cm diameter,PTPEJQR9VKFUV4HY,Key Features of oxford Tawa 15 cm cm diameter ...
